# Thu thập và lưu ảnh khuôn mặt

Notebook này dùng để thu thập ảnh khuôn mặt của sinh viên và lưu vào dataset.

## 1. Cài đặt thư viện

In [ ]:
# Cài đặt các thư viện cần thiết
!pip install opencv-python
!pip install opencv-python-headless
!pip install pillow

## 2. Mount Google Drive (nếu dùng Colab)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Đường dẫn đến thư mục dự án trên Drive
PROJECT_PATH = '/content/drive/MyDrive/face_attendance'

import os
os.chdir(PROJECT_PATH)

## 3. Import thư viện

In [ ]:
import cv2
import os
import numpy as np
from PIL import Image
from google.colab.patches import cv2_imshow
from IPython.display import display, Javascript
from google.colab import output
from base64 import b64decode

## 4. Cấu hình

In [ ]:
# Cấu hình
DATASET_DIR = 'dataset'
HAARCASCADE_PATH = 'haarcascades/haarcascade_frontalface_default.xml'
IMAGES_PER_PERSON = 50
IMAGE_SIZE = (160, 160)

# Tạo thư mục nếu chưa có
os.makedirs(DATASET_DIR, exist_ok=True)
os.makedirs('haarcascades', exist_ok=True)

## 5. Tải Haar Cascade

In [ ]:
# Tải Haar Cascade nếu chưa có
if not os.path.exists(HAARCASCADE_PATH):
    !wget https://raw.githubusercontent.com/opencv/opencv/master/data/haarcascades/haarcascade_frontalface_default.xml -P haarcascades/
    print("Đã tải Haar Cascade")
else:
    print("Haar Cascade đã tồn tại")

## 6. Hàm thu thập ảnh từ webcam

In [ ]:
def take_photo(filename='photo.jpg', quality=0.8):
    """
    Chụp ảnh từ webcam trong Google Colab
    """
    js = Javascript('''
    async function takePhoto(quality) {
      const div = document.createElement('div');
      const capture = document.createElement('button');
      capture.textContent = 'Chụp ảnh';
      div.appendChild(capture);

      const video = document.createElement('video');
      video.style.display = 'block';
      const stream = await navigator.mediaDevices.getUserMedia({video: true});

      document.body.appendChild(div);
      div.appendChild(video);
      video.srcObject = stream;
      await video.play();

      // Resize the output to fit the video element.
      google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);

      // Wait for Capture to be clicked.
      await new Promise((resolve) => capture.onclick = resolve);

      const canvas = document.createElement('canvas');
      canvas.width = video.videoWidth;
      canvas.height = video.videoHeight;
      canvas.getContext('2d').drawImage(video, 0, 0);
      stream.getVideoTracks()[0].stop();
      div.remove();
      return canvas.toDataURL('image/jpeg', quality);
    }
    ''')
    display(js)
    data = eval_js('takePhoto({})'.format(quality))
    binary = b64decode(data.split(',')[1])
    
    with open(filename, 'wb') as f:
        f.write(binary)
    
    return filename

from google.colab import output
from IPython.display import Javascript

def eval_js(code):
    return output.eval_js(code)

## 7. Thu thập ảnh cho một người

In [ ]:
def collect_images_for_person(person_name, num_images=IMAGES_PER_PERSON):
    """
    Thu thập ảnh cho một người
    """
    # Tạo thư mục cho người này
    person_dir = os.path.join(DATASET_DIR, person_name)
    os.makedirs(person_dir, exist_ok=True)
    
    # Load Haar Cascade
    face_cascade = cv2.CascadeClassifier(HAARCASCADE_PATH)
    
    print(f"Bắt đầu thu thập {num_images} ảnh cho {person_name}")
    print("Nhấn nút 'Chụp ảnh' để chụp mỗi ảnh")
    
    count = 0
    
    while count < num_images:
        try:
            # Chụp ảnh
            print(f"\nẢnh {count + 1}/{num_images}")
            filename = take_photo(f'temp_{count}.jpg')
            
            # Đọc ảnh
            image = cv2.imread(filename)
            gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
            
            # Phát hiện khuôn mặt
            faces = face_cascade.detectMultiScale(gray, 1.1, 5, minSize=(30, 30))
            
            if len(faces) > 0:
                # Lấy khuôn mặt đầu tiên
                (x, y, w, h) = faces[0]
                face = image[y:y+h, x:x+w]
                
                # Resize
                face = cv2.resize(face, IMAGE_SIZE)
                
                # Lưu ảnh
                save_path = os.path.join(person_dir, f"{person_name}_{count}.jpg")
                cv2.imwrite(save_path, face)
                
                # Hiển thị
                print("✓ Đã lưu ảnh")
                cv2_imshow(face)
                
                count += 1
            else:
                print("✗ Không phát hiện khuôn mặt, vui lòng thử lại")
            
            # Xóa file tạm
            os.remove(filename)
            
        except Exception as e:
            print(f"Lỗi: {str(e)}")
    
    print(f"\n✓ Hoàn thành thu thập {count} ảnh cho {person_name}")

## 8. Chạy thu thập ảnh

In [ ]:
# Nhập tên sinh viên
person_name = input("Nhập tên sinh viên: ").strip()

if person_name:
    collect_images_for_person(person_name)
else:
    print("Tên không được để trống!")

## 9. Xem danh sách người đã thu thập

In [ ]:
# Xem danh sách
persons = [d for d in os.listdir(DATASET_DIR) if os.path.isdir(os.path.join(DATASET_DIR, d))]

print(f"Tổng số người: {len(persons)}")
for i, person in enumerate(persons, 1):
    person_dir = os.path.join(DATASET_DIR, person)
    num_images = len([f for f in os.listdir(person_dir) if f.endswith('.jpg')])
    print(f"{i}. {person} - {num_images} ảnh")